# Versione Colab del prompting engineering di **QWEN**

## necessari install

In [1]:
!pip install -U transformers

In [2]:
!pip install qwen-vl-utils[decord]==0.0.8

## per installare il dataset

In [3]:
!pip install kaggle

In [4]:
from google.colab import files
uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [5]:
import os
import shutil

# Crea la cartella corretta
os.makedirs("/root/.config/kaggle", exist_ok=True)

# Sposta il file kaggle.json se è nella directory corrente
if os.path.exists("kaggle.json"):
    shutil.move("kaggle.json", "/root/.config/kaggle/kaggle.json")

# Permessi corretti
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)


In [6]:
import kaggle
kaggle.api.authenticate()
kaggle.api.dataset_download_files("xhlulu/140k-real-and-fake-faces", path=".", unzip=True)

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces


## Local Inference on GPU
Model page: https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

## Codice per testare il prompting delle immagini

### import

In [3]:
import re
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json
from datetime import datetime
import torch

### caricare il modello **Qwen2.5-VL-3B**

In [4]:
# Load model directly
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, AutoTokenizer
from qwen_vl_utils import process_vision_info

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype="auto", device_map="auto"
)

#inputs = processor.apply_chat_template(
#	messages,
#	add_generation_prompt=True,
#	tokenize=True,
#	return_dict=True,
#	return_tensors="pt",
#).to(model.device)

#outputs = model.generate(**inputs, max_new_tokens=40)
#print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

### Codice di prompt testing

In [ ]:
def chooseAPrompt(index, language):
    if language:
        Prompt = PromptListIta[index]
    else:
        Prompt = PromptListEng[index]
    return Prompt


# lista di prompt da scegliere
PromptListEng = ["Is the provided image showing a real face or a generated one?",  # 0
                 "Is the provided image showing a real face or a generated one? Keep in mind that generated faces often"
                 "exhibit artifacts around the mouth, ears, and hairline. Inconsistent lighting and shading may also be"
                 "present.",  # 1
                 "Are there any artifacts around the mouth, ears, or hairline that might indicate the face is "
                 "AI-generated?",  # 2
                 "Is the provided image showing a real face or a generated one? I will pay you $1,000 for the correct "
                 "answer.",  # 3
                 "This is a life-threatening situation. This person is blackmailing me, and I need to know whether "
                 "this is a real person or an AI-generated face.",  # 4
                 "I believe this person is real. Based on the image, does it appear to be a real face or a generated "
                 "one?",  # 5
                 "I believe this person doesn’t exist. Based on the image, does it appear to be a real face or a "
                 "generated one?"  # 6
                 ]
PromptListIta = [
    "L'immagine fornita mostra un volto reale o uno generato?",
    "L'immagine fornita mostra un volto reale o uno generato? Tieni presente che i volti generati spesso presentano "
    "artefatti intorno alla bocca, alle orecchie e all'attaccatura dei capelli. Inoltre, potrebbero esserci "
    "condizioni di illuminazione e ombreggiatura incoerenti.",
    "Ci sono artefatti intorno alla bocca, alle orecchie o all'attaccatura dei capelli che potrebbero indicare che il "
    "volto è stato generato dall'IA?",
    "L'immagine fornita mostra un volto reale o uno generato? Ti pagherò 1.000 dollari per una risposta corretta.",
    "Questa è una situazione che mette a rischio la mia vita. Questa persona mi sta ricattando e ho bisogno di sapere "
    "se si tratta di una persona reale o di un volto generato dall'intelligenza artificiale.",
    "Credo che questa persona sia reale. In base all'immagine, ti sembra un volto reale o uno generato?",
    "Credo che questa persona non esista. In base all'immagine, ti sembra un volto reale o uno generato?"
]
IndexPrompt = 0  # scelgo io in base al prompt che voglio (0-6)
PromptITA = True  # scelgo il linguaggio che voglio

PromptP1 = chooseAPrompt(IndexPrompt, PromptITA)
# reinforcement con un prompt system + prompt user
MODEL_NAME = "qwen2.5"
PROMPT = (
        PromptP1 +
        # JSON output
        " You must respond using exactly and only this structured JSON format:\n\n"
        "{\n"
        "  \"result\": \"[real face/generated/uncertain]\",\n"
        "  \"explanation\": \"[a long explanation clearly stating the reasoning behind your choice]\"\n"
        "}\n\n"
        "Do not add any commentary outside this JSON format."
)

MAX_IMAGES_PER_CLASS = 50
SHOW_IMAGES = False
# ================

# Dataset paths
fake_dir = Path("real_vs_fake/real-vs-fake/test/fake")
real_dir = Path("real_vs_fake/real-vs-fake/test/real")

# Load images
fake_images = list(fake_dir.glob("*.jpg"))[:MAX_IMAGES_PER_CLASS]
real_images = list(real_dir.glob("*.jpg"))[:MAX_IMAGES_PER_CLASS]
images_with_labels = [(img, 1) for img in real_images] + [(img, 0) for img in fake_images]
print("You choose this: " + PROMPT + "\n")


# Initialize counters
counters = {
    "tp": 0, "tn": 0, "fp": 0, "fn": 0, "er": 0,
    "rejection_real": 0, "rejection_fake": 0
}


def analyze_image(img_path, lab):
    try:
        messages = [
          {
              "role": "system",
              "content": [
                  {
                      "type": "text",
                      "text": (
                          "You are a professional image classification AI. Given an image, determine if it depicts a real human face or a generated one. "
                          "Respond ONLY with a JSON object containing two fields:\n"
                          "{\n"
                          "  \"result\": \"[real face]\" or \"[generated face]\",\n"
                          "  \"explanation\": \"A clear, concise explanation highlighting key visual cues such as skin texture, facial features, lighting, or artifacts that support your decision.\"\n"
                          "}\n"
                          "Do NOT include anything else in your response. If uncertain, classify as \"[fake face]\" and explain why."
                      )
                  }
              ]
          },
          {"role": "user", "content": [{"type": "image", "image": str(img_path)}, {"type": "text", "text": PROMPT}]}
      ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")

        generated_ids = model.generate(**inputs, max_new_tokens=128)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )

        print(f"\nImage: {img_path.name}")
        print("Raw Output:", output_text)
        # Metodo per fare il parsing usando l'output JSON
        text_raw = output_text[0].strip()
        # Clean potential markdown fences
        text_clean = re.sub(r"^```(?:json)?\s*([\s\S]*?)\s*```$", r"\1", text_raw.strip(), flags=re.MULTILINE)
        try:
            parsed = json.loads(text_clean)
            result = parsed.get("result")

            # Gestione di valori tipo lista o altro
            if isinstance(result, list) and result:
                result = result[0]
            prediction = str(result).strip().lower()
        except Exception as e:
            counters["er"] += 1
            print(f" JSON Parsing error: {e}")
            return
        # parsing se non si usa il JSON, non è molto efficace
        # match = re.search(r"(?:\[)?(yes|no|uncertain)(?:\])?", text.lower())
        # if not match:
        #     counters["er"] += 1
        #     print(" Parsing error, no match.")
        #     return
        #
        # prediction = match.group(1)

        if SHOW_IMAGES:
            Image.open(img_path).show()

        if lab == 1:  # Real
            if (prediction == "real face" or prediction == "real" or prediction == "[real face]" or
                    prediction == "agreed" or prediction == "[real]"):
                counters["tn"] += 1
                print(" TN (real correctly identified)")
            elif prediction == "generated" or prediction == "[generated]" or prediction == "didn't agree":
                counters["fp"] += 1
                print(" FP (real misclassified as fake)")
            else:  # uncertain
                counters["rejection_real"] += 1
                print(" Rejection on real image")
        else:  # Fake
            if prediction == "generated" or prediction == "[generated]" or prediction == "didn't agree":
                counters["tp"] += 1
                print(" TP (fake correctly identified)")  # diamo importanza all'identificare il falso adesso
            elif (prediction == "real face" or prediction == "real" or prediction == "[real face]" or
                  prediction == "agreed" or prediction == "[real]"):
                counters["fn"] += 1
                print(" FN (fake misclassified as real)")
            else:  # uncertain
                counters["rejection_fake"] += 1
                print(" Rejection on fake image")

    except Exception as e:
        print(f" Error on {img_path}: {e}")
        counters["er"] += 1


# Main analysis loop
for img_path, label in tqdm(images_with_labels, desc=" Analyzing images"):
    analyze_image(img_path, label)

# Metrics
total_classified = counters["tp"] + counters["tn"] + counters["fp"] + counters["fn"]
accuracy = (counters["tp"] + counters["tn"]) / total_classified if total_classified else 0
precision = counters["tp"] / (counters["tp"] + counters["fp"]) if (counters["tp"] + counters["fp"]) else 0
recall = counters["tp"] / (counters["tp"] + counters["fn"]) if (counters["tp"] + counters["fn"]) else 0

total_real = counters["tp"] + counters["fn"] + counters["rejection_real"]
total_fake = counters["tn"] + counters["fp"] + counters["rejection_fake"]

rejection_real_rate = counters["rejection_real"] / total_real if total_real else 0
rejection_fake_rate = counters["rejection_fake"] / total_fake if total_fake else 0
rejection_total_rate = (counters["rejection_real"] + counters["rejection_fake"]) / (total_real + total_fake)

false_negative_rate = counters["fn"] / total_real if total_real else 0
false_positive_rate = counters["fp"] / total_fake if total_fake else 0

# Results
print("\n====== FINAL REPORT ======")
print(f"Total processed: {len(images_with_labels)}")
print(f"TP: {counters['tp']} | TN: {counters['tn']} | FP: {counters['fp']} | FN: {counters['fn']}")
print(f"Rejections on real: {counters['rejection_real']} | Rejections on fake: {counters['rejection_fake']}")
print(f"Text parsing errors: {counters['er']} ({(counters['er'] / len(images_with_labels)) * 100:.2f}%)\n")

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"False Negative Rate (real->fake): {false_negative_rate * 100:.2f}%")
print(f"False Positive Rate (fake->real): {false_positive_rate * 100:.2f}%")
print(f"Rejection Rate on real images: {rejection_real_rate * 100:.2f}%")
print(f"Rejection Rate on fake images: {rejection_fake_rate * 100:.2f}%")

# salvataggio in json
results = {
    "total_processed": len(images_with_labels),
    "total_real": len(real_images),
    "total_fake": len(fake_images),
    "TP": counters["tp"],
    "TN": counters["tn"],
    "FP": counters["fp"],
    "FN": counters["fn"],
    "rejection_real": counters["rejection_real"],
    "rejection_fake": counters["rejection_fake"],
    "text_parsing_errors": counters["er"],
    "text_parsing_error_rate": (counters["er"] / len(images_with_labels)) if len(images_with_labels) else 0,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "false_negative_rate": false_negative_rate,
    "false_positive_rate": false_positive_rate,
    "rejection_real_rate": rejection_real_rate,
    "rejection_fake_rate": rejection_fake_rate,
    "rejection_total_rate": rejection_total_rate,
}

# Crea cartella se non esiste
Path("resultsJSON").mkdir(exist_ok=True)

# Imposta lingua
language_tag = "ITA" if PromptITA else "ENG"

# Timestamp per identificare diversi tentativi
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

# Pulisci MODEL_NAME da caratteri non ammessi nei nomi file
safe_model_name = MODEL_NAME.replace(":", "_").replace("/", "_")

# Costruisci filename
filename = f"resultsJSON/real-vs-fake_{safe_model_name}_PromptType-{IndexPrompt}_{language_tag}_{timestamp}_result.json"

# Salva JSON
with open(filename, "w") as f:
    json.dump(results, f, indent=4)

print(f"Results saved to {filename}.")

You choose this: L'immagine fornita mostra un volto reale o uno generato? You must respond using exactly and only this structured JSON format:

{
  "result": "[real face/generated/uncertain]",
  "explanation": "[a long explanation clearly stating the reasoning behind your choice]"
}

Do not add any commentary outside this JSON format.



 Analyzing images:   1%|          | 1/100 [00:06<10:01,  6.07s/it]


Image: 13393.jpg
Raw Output: ['{\n  "result": "[real face]",\n  "explanation": "The image shows a person speaking into a microphone, which is a common scenario for real people. The facial features, including the skin texture, hair, and beard, appear natural and consistent with real human characteristics. There are no visible artifacts or anomalies that would suggest the image is generated."\n}']
 TN (real correctly identified)


 Analyzing images:   2%|▏         | 2/100 [00:10<08:23,  5.14s/it]


Image: 34812.jpg
Raw Output: ['{\n  "result": "[real face]",\n  "explanation": "The image shows a clear and detailed human face with natural skin texture, realistic facial features, and even lighting that suggests a real photograph rather than a generated image. There are no visible artifacts or anomalies that would indicate it is a generated face."\n}']
 TN (real correctly identified)


...


Image: 11216.jpg
Raw Output: ['{\n  "result": "[real face]",\n  "explanation": "The image shows a clear and detailed human face with natural skin texture, realistic facial features, and a well-lit environment. There are no visible artifacts or anomalies that would suggest it is a generated image."\n}']
 TN (real correctly identified)


 Analyzing images:  17%|█▋        | 17/100 [01:28<06:35,  4.76s/it]


Image: 03004.jpg
Raw Output: ['{\n  "result": "[real face]",\n  "explanation": "The image shows a person with a beard and sunglasses, which suggests a real human face. The skin texture appears natural, and there are no visible artifacts or signs of digital manipulation that would indicate a generated face."\n}']
 TN (real correctly identified)


 Analyzing images:  18%|█▊        | 18/100 [01:32<06:12,  4.54s/it]


Image: 39739.jpg
Raw Output: ['{\n  "result": "[real face]",\n  "explanation": "The image shows a person with visible skin texture, natural facial features, and lighting that suggests a real photograph. There are no obvious signs of digital manipulation or artificial elements that would indicate a generated face."\n}']
 TN (real correctly identified)


In [13]:
import torch
import gc

# Elimina tutte le variabili Python inutilizzate
gc.collect()

# Libera la cache CUDA
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


## per scaricare result in JSON

In [1]:
import shutil
from google.colab import files

# Sostituisci con il percorso della tua cartella
folder_path = "/content/resultsJSON"
zip_path = "/content/resultsJSON.zip"

# Comprimi la cartella
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', folder_path)

# Scarica il file zip
files.download(zip_path)


FileNotFoundError: [Errno 2] No such file or directory: '/content/resultsJSON'